In [ ]:
pip install shap
pip install kneed
pip install kmodes
pip install prince
pip install stepmix
pip install openpyxl

In [ ]:
import shap
from google.colab import files
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Matplotlib is a plotting library for python and pyplot gives us a MatLab like plotting framework. We will use this in our plotter function to plot data.
import matplotlib.pyplot as plt
#Seaborn is a Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import seaborn as sns
# Preprocessing allows us to standarsize our data
from sklearn import preprocessing
# Allows us to split our data into training and testing data
from sklearn.model_selection import train_test_split
# Allows us to test parameters of classification algorithms and find the best one
from sklearn.model_selection import GridSearchCV
# Logistic Regression classification algorithm
from sklearn.linear_model import LogisticRegression
# Support Vector Machine classification algorithm
from sklearn.svm import SVC
# Decision Tree classification algorithm
from sklearn.tree import DecisionTreeClassifier
# K Nearest Neighbors classification algorithm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [ ]:
# Upload your CSV from local
uploaded = files.upload()

Saving question.xlsx to question (1).xlsx
Saving ResultsForSurvey-398 - 07-13-2026 08_17_55.xlsx to ResultsForSurvey-398 - 07-13-2026 08_17_55 (1).xlsx


In [ ]:
# Load it into a DataFrame
df = pd.read_excel("ResultsForSurvey-398 - 07-13-2026 08_17_55.xlsx")
df_pitanja = pd.read_excel("question.xlsx")

In [ ]:
# Question IDs to keep
keep_cols = [#ids you want to keep
             ]

# Check which columns exist
existing = [c for c in keep_cols if c in df.columns]
missing = [c for c in keep_cols if c not in df.columns]

print("="*50)
print(f"Columns requested: {len(keep_cols)}")
print(f"Columns found:     {len(existing)}")
print(f"Columns missing:   {len(missing)}")
print("="*50)

print("\n✓ Existing columns:")
for c in existing:
    print(c)

if missing:
    print("\n✗ Missing columns:")
    for c in missing:
        print(c)

# Keep only these columns
df = df[existing].copy()

print("\nNew dataframe shape:", df.shape)

In [ ]:
ordinal_vars = []

for qid, qtype, options in zip(df_pitanja["QuestionID"], df_pitanja["Type"], df_pitanja["Options"]):
    if qtype == "Scale" and pd.notnull(options):
        choices = [opt.strip() for opt in str(options).split(';') if '=' in opt]
        if len(choices) > 2:   # more than 2 options
            ordinal_vars.append(str(qid))

# same cleanup you used (remove .0)
ordinal_vars = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in ordinal_vars]

In [ ]:
to_remove = [#ids
]

ordinal_vars = [v for v in ordinal_vars if v not in to_remove]

In [ ]:
select_one = []
for qid, qtype in zip(df_pitanja["QuestionID"], df_pitanja["Type"]):
    if qtype == "SelectOne":
        select_one.append(qid)
select_one = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in select_one]

In [ ]:
select_one.extend(to_remove)

In [ ]:
multi_nominal_cols = []
for qid, qtype in zip(df_pitanja["QuestionID"], df_pitanja["Type"]):
    if qtype == "SelectMultiple":
        multi_nominal_cols.append(qid)
multi_nominal_cols = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in multi_nominal_cols]

In [ ]:
binary_cols = [#ids
]

select_one = [v for v in select_one if v not in binary_cols]

In [ ]:
binary = []

for qid, qtype, options in zip(df_pitanja["QuestionID"], df_pitanja["Type"], df_pitanja["Options"]):
    if qtype == "Scale" and pd.notnull(options):
        choices = [opt.strip() for opt in str(options).split(';') if '=' in opt]
        if len(choices) <= 2:   # <= 2 options
            binary.append(str(qid))

binary = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in binary]

In [ ]:
# See missing values per column
df_mising=df.isnull().sum().sort_values(ascending=False)

In [ ]:
df_mising

In [ ]:
za_drop = []

for index, value in df_mising.items():
    if value > 600:
        za_drop.append(index)

In [ ]:
za_drop = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in za_drop]

In [ ]:
za_drop2 = []

for qid, qtype in zip(df_pitanja["QuestionID"], df_pitanja["Type"]):
    if qtype == "Text":
        za_drop2.append(str(qid))  # convert to string here


In [ ]:
za_drop2_cleaned = [str(int(float(col))) if str(col).endswith('.0') else str(col) for col in za_drop2]

In [ ]:
df = df.drop(columns=za_drop)

In [ ]:
df = df.drop(columns=za_drop2_cleaned, errors='ignore')


In [ ]:
demographic_cols = [
#ids
]


In [ ]:
# Step 2: Filter out columns that don’t exist in the DataFrame
ordinal_vars = [col for col in ordinal_vars if col in df.columns]

# Step 3: Convert only valid ones
for col in ordinal_vars:
    df[col] = df[col].astype('Int64')


In [ ]:
demographic_cols = [
#ids
]

# Remove demographics from preprocessing lists
select_one = [c for c in select_one if c not in demographic_cols]
ordinal_vars = [c for c in ordinal_vars if c not in demographic_cols]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# =====================================================
# MULTI-SELECT questions
# =====================================================
for col in multi_nominal_cols:
    if col in df.columns:
        raw = df[col].fillna('')
        dummies = raw.astype(str).str.get_dummies(sep=';')
        if '' in dummies.columns:
            dummies = dummies.drop(columns=[''])
        dummies.columns = [f"{col}_{c.strip()}" for c in dummies.columns]
        df[f'{col}_no_selection'] = (dummies.sum(axis=1) == 0).astype(int)
        df = df.drop(columns=[col])
        df = pd.concat([df, dummies], axis=1)

# =====================================================
# BINARY questions — single categorical column
# =====================================================
for col in binary:
    if col in df.columns:
        df[col] = df[col].replace({1: "yes", 2: "no"})

for col in [#ids
            ]:
    if col in df.columns:
        df[col] = df[col].replace('', np.nan).fillna("not_applicable")

# =====================================================
# Demographic and ordinal column lists — defined ONCE
# =====================================================
demographic_nominal = [#ids
                       ]
demographic_ordinal = [#ids
                       ]
true_ordinal_cols = [
    #ids
]

structural_missing_cols = [#ids
                           ]
random_missing_cols = [c for c in select_one if c not in structural_missing_cols]

# =====================================================
# SELECT-ONE — single categorical column
# =====================================================
for col in select_one:
    if col in df.columns:
        cat = df[col].replace('', np.nan)
        if col in structural_missing_cols:
            cat = cat.fillna("not_applicable")
        else:
            mode_val = cat.mode(dropna=True)[0]
            cat = cat.fillna(mode_val)
        df[col] = cat

# GENDER — single categorical column
if "id" in df.columns:
    df["id"] = df["id"].replace('', np.nan).fillna("not_disclosed")

# =====================================================
# Build df_proc: coerce ONLY true ordinal columns to numeric
# =====================================================
df_proc = df.copy()

for col in true_ordinal_cols + demographic_ordinal:
    if col in df_proc.columns:
        df_proc[col] = pd.to_numeric(df_proc[col], errors="coerce")

ordinal_cols = [c for c in true_ordinal_cols + demographic_ordinal if c in df_proc.columns]

if ordinal_cols:
    imputer = SimpleImputer(strategy="median")
    df_proc[ordinal_cols] = imputer.fit_transform(df_proc[ordinal_cols])

print("Missing values remaining:", df_proc.isna().sum().sum())

# =====================================================
# Build X_full: scale ordinal, leave categorical raw
# =====================================================
categorical_cols = [c for c in df_proc.columns if c not in ordinal_cols]

scaler = StandardScaler()
ordinal_scaled = pd.DataFrame(
    scaler.fit_transform(df_proc[ordinal_cols]),
    columns=ordinal_cols,
    index=df_proc.index
)

categorical_raw = df_proc[categorical_cols].copy()

X_full = pd.concat([ordinal_scaled, categorical_raw], axis=1)
print("Ordinal:", len(ordinal_cols), "| Categorical:", len(categorical_cols))

In [ ]:
import pandas as pd
import numpy as np
from kmodes.kprototypes import KPrototypes
from kmodes.kmodes import KModes
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# =====================================================
# FINAL DIMENSION STRUCTURE
# =====================================================

cluster_models = {
    "Advice Style": [#ids
                     ],
    "Advice Topics of Interest": [#ids
                                  ],
    "Channel Behavior & Preference": [#ids
                                      ],
    "Digital & Fintech Engagement": [#ids
                                     ],
    "Information & Contact Openness": [#ids
                                       ],
    "Membership Engagement": [#ids
                              ],
}

# Method to use per dimension, based on its variable composition
cluster_method_map = {
    "Advice Style": "kprototypes",              # mixed ordinal + categorical
    "Advice Topics of Interest": "kmodes",       # pure categorical (multi-select)
    "Channel Behavior & Preference": "kprototypes",  # mixed ordinal + categorical
    "Digital & Fintech Engagement": "kmodes",    # pure categorical
    "Information & Contact Openness": "kmodes",  # pure categorical
    "Membership Engagement": "kmodes",           # pure categorical, expect 2-tier structure
}

# =====================================================
# BUILD DATASETS (with mixed-type fix baked in)
# =====================================================

cluster_datasets = {}
categorical_index_map = {}

for model_name, prefixes in cluster_models.items():
    selected_columns = []
    for prefix in prefixes:
        matches = [c for c in X_full.columns if c == prefix or c.startswith(prefix + "_")]
        selected_columns.extend(matches)

    dataset = X_full[selected_columns].copy()

    # Force categorical columns to string to avoid the mixed str/float crash
    for c in dataset.columns:
        if c in categorical_cols:
            dataset[c] = dataset[c].astype(str)

    cluster_datasets[model_name] = dataset

    cat_positions = [i for i, c in enumerate(dataset.columns) if c in categorical_cols]
    categorical_index_map[model_name] = cat_positions

    print(f"{model_name}: {dataset.shape[1]} columns, {len(cat_positions)} categorical "
          f"-> method: {cluster_method_map[model_name]}")


In [ ]:
from kmodes.kprototypes import KPrototypes
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    contingency = pd.crosstab(x, y)
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    return (chi2 / (n * min_dim)) ** 0.5

advice_vars = [#ids
               ]
advice_dataset = cluster_datasets["Advice Style"]
cat_idx = categorical_index_map["Advice Style"]

# See what gamma the default run auto-computed
default_model = KPrototypes(n_clusters=5, init="Cao", random_state=42, n_init=10)
default_model.fit_predict(advice_dataset.to_numpy(), categorical=cat_idx)
print("Default auto-computed gamma:", round(default_model.gamma, 3))

gamma_values = sorted(set([0.1, 0.25, 0.5, round(default_model.gamma, 3), 1.0, 2.0, 4.0]))

gamma_test_results = []
for gamma in gamma_values:
    model = KPrototypes(n_clusters=5, init="Cao", random_state=42, n_init=10, gamma=gamma)
    labels = model.fit_predict(advice_dataset.to_numpy(), categorical=cat_idx)
    row = {"gamma": gamma, "cost": round(model.cost_, 1)}
    for v in advice_vars:
        row[f"CramersV_{v}"] = round(cramers_v(labels, df_proc[v]), 3)
    gamma_test_results.append(row)

gamma_test_df = pd.DataFrame(gamma_test_results)
print(gamma_test_df.to_string(index=False))

In [ ]:
from kmodes.kprototypes import KPrototypes
import matplotlib.pyplot as plt
import pandas as pd

advice_dataset = cluster_datasets["Advice Style"]
cat_idx = categorical_index_map["Advice Style"]

gamma_values = [0.1, 0.25, 0.5, 1.0, 2.0, 4.0]
k_range = range(2, 8)

elbow_results = []
for gamma in gamma_values:
    for k in k_range:
        model = KPrototypes(n_clusters=k, init="Cao", random_state=42, n_init=10, gamma=gamma)
        model.fit_predict(advice_dataset.to_numpy(), categorical=cat_idx)
        elbow_results.append({"gamma": gamma, "k": k, "cost": model.cost_})
        print(f"gamma={gamma}, k={k}: cost={model.cost_:.1f}")

elbow_df = pd.DataFrame(elbow_results)

plt.figure(figsize=(8, 5))
for gamma in gamma_values:
    subset = elbow_df[elbow_df["gamma"] == gamma]
    plt.plot(subset["k"], subset["cost"], marker="o", label=f"gamma={gamma}")

plt.xlabel("k")
plt.ylabel("Cost (lower is better)")
plt.title("Advice Style — KPrototypes elbow curve across gamma values")
plt.legend()
plt.show()

In [ ]:
# =====================================================
# GENERALIZED, AUTOMATED PER-DIMENSION PIPELINE
# Requires cluster_models, cluster_method_map, cluster_datasets,
# categorical_index_map, df_proc to already exist (from your
# original setup cell) before this runs.
# =====================================================

# pip install kneed --quiet   # run once if not already installed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from kmodes.kprototypes import KPrototypes
from kmodes.kmodes import KModes
from sklearn.cluster import KMeans
from stepmix.stepmix import StepMix
from sklearn.preprocessing import LabelEncoder
from scipy.stats import chi2_contingency

try:
    from kneed import KneeLocator
    HAVE_KNEED = True
except ImportError:
    HAVE_KNEED = False
    print("kneed not installed (pip install kneed) — using a simple fallback elbow heuristic.")


def cramers_v(x, y):
    contingency = pd.crosstab(x, y)
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    return (chi2 / (n * min_dim)) ** 0.5


def cluster_balance(labels):
    counts = pd.Series(labels).value_counts(normalize=True)
    k = len(counts)
    ent = -(counts * np.log(counts)).sum()
    return ent / np.log(k) if k > 1 else 0.0


def detect_elbow(k_values, costs):
    if HAVE_KNEED:
        kl = KneeLocator(k_values, costs, curve="convex", direction="decreasing")
        return kl.elbow
    diffs2 = np.diff(np.diff(np.array(costs)))
    if len(diffs2) == 0:
        return None
    return k_values[np.argmax(diffs2) + 1]


def run_dimension(dimension_name, k_range=range(2, 8), entropy_floor=0.5, fallback_k=5,
                   show_plot=True):
    print("=" * 70)
    print(f"DIMENSION: {dimension_name}")
    print("=" * 70)

    dataset = cluster_datasets[dimension_name]
    method = cluster_method_map[dimension_name]
    cat_idx = categorical_index_map[dimension_name]
    raw_cols = cluster_models[dimension_name]

    # -----------------------------------------------------------
    # A. Traditional method — KPrototypes, KModes, or KMeans
    #    k selected via elbow detection on the cost/inertia curve
    # -----------------------------------------------------------
    trad_costs = []
    for k in k_range:
        if method == "kprototypes":
            m = KPrototypes(n_clusters=k, init="Cao", random_state=42, n_init=10)
            m.fit_predict(dataset.to_numpy(), categorical=cat_idx)
            trad_costs.append(m.cost_)
        elif method == "kmodes":
            m = KModes(n_clusters=k, init="Cao", random_state=42, n_init=10)
            m.fit_predict(dataset.to_numpy())
            trad_costs.append(m.cost_)
        elif method == "kmeans":
            m = KMeans(n_clusters=k, random_state=42, n_init=10)
            m.fit_predict(dataset.to_numpy())
            trad_costs.append(m.inertia_)
        else:
            raise ValueError(f"Unknown method '{method}' for dimension '{dimension_name}'")

    trad_elbow_k = detect_elbow(list(k_range), trad_costs)
    trad_k = trad_elbow_k if trad_elbow_k is not None else fallback_k

    if method == "kprototypes":
        trad_final = KPrototypes(n_clusters=trad_k, init="Cao", random_state=42, n_init=15)
        trad_labels = trad_final.fit_predict(dataset.to_numpy(), categorical=cat_idx)
        trad_native_metric = f"cost={trad_final.cost_:.1f}"
    elif method == "kmodes":
        trad_final = KModes(n_clusters=trad_k, init="Cao", random_state=42, n_init=15)
        trad_labels = trad_final.fit_predict(dataset.to_numpy())
        trad_native_metric = f"cost={trad_final.cost_:.1f}"
    elif method == "kmeans":
        trad_final = KMeans(n_clusters=trad_k, random_state=42, n_init=15)
        trad_labels = trad_final.fit_predict(dataset.to_numpy())
        trad_native_metric = f"inertia={trad_final.inertia_:.1f}"

    trad_series = pd.Series(trad_labels, index=dataset.index)

    if show_plot:
        plt.figure(figsize=(6, 4))
        plt.plot(list(k_range), trad_costs, marker="o")
        if trad_elbow_k is not None:
            plt.axvline(trad_elbow_k, color="red", linestyle="--", label=f"elbow k={trad_elbow_k}")
            plt.legend()
        plt.xlabel("k")
        plt.ylabel("Cost / inertia")
        plt.title(f"{dimension_name} — {method} cost curve")
        plt.show()

    # -----------------------------------------------------------
    # B. LCA — every column treated as categorical, k via BIC
    #    (ordinal columns are label-encoded like any other; see
    #    the Advice Style writeup for why this is the safer choice)
    # -----------------------------------------------------------
    lca_raw = dataset.copy()
    for c in lca_raw.columns:
        lca_raw[c] = LabelEncoder().fit_transform(lca_raw[c].astype(str))
    X_lca = lca_raw.to_numpy()

    lca_scan = []
    for k in k_range:
        m = StepMix(n_components=k, measurement="categorical", random_state=42, n_init=15)
        m.fit(X_lca)
        lca_scan.append({"k": k, "bic": m.bic(X_lca), "aic": m.aic(X_lca),
                          "entropy": m.relative_entropy(X_lca)})
    lca_scan_df = pd.DataFrame(lca_scan)
    lca_best_k = int(lca_scan_df.loc[lca_scan_df["bic"].idxmin(), "k"])

    if show_plot:
        plt.figure(figsize=(6, 4))
        plt.plot(lca_scan_df["k"], lca_scan_df["bic"], marker="o", label="BIC")
        plt.plot(lca_scan_df["k"], lca_scan_df["aic"], marker="o", label="AIC")
        plt.axvline(lca_best_k, color="red", linestyle="--", label=f"BIC min k={lca_best_k}")
        plt.xlabel("k")
        plt.ylabel("Information criterion")
        plt.title(f"{dimension_name} — LCA model selection")
        plt.legend()
        plt.show()

    lca_final = StepMix(n_components=lca_best_k, measurement="categorical",
                         random_state=42, n_init=25)
    lca_final.fit(X_lca)
    lca_labels = lca_final.predict(X_lca)
    lca_series = pd.Series(lca_labels, index=lca_raw.index)
    lca_entropy = lca_final.relative_entropy(X_lca)
    lca_native_metric = f"BIC={lca_final.bic(X_lca):.1f}, entropy={lca_entropy:.3f}"

    # -----------------------------------------------------------
    # C. Comparison table (Cramer's V per raw variable, both methods)
    # -----------------------------------------------------------
    comparison_rows = [
        {
            "Method": method,
            "k": trad_series.nunique(),
            **{f"CramersV_{v}": round(cramers_v(trad_series, df_proc[v]), 3)
               for v in raw_cols if v in df_proc.columns},
            "Cluster_balance": round(cluster_balance(trad_series), 3),
            "Native_fit_metric": trad_native_metric,
        },
        {
            "Method": "LCA",
            "k": lca_series.nunique(),
            **{f"CramersV_{v}": round(cramers_v(lca_series, df_proc[v]), 3)
               for v in raw_cols if v in df_proc.columns},
            "Cluster_balance": round(cluster_balance(lca_series), 3),
            "Native_fit_metric": lca_native_metric,
        },
    ]
    comparison_df = pd.DataFrame(comparison_rows)
    print("\nMethod comparison:")
    print(comparison_df.to_string(index=False))

    # -----------------------------------------------------------
    # D. Automated decision
    # -----------------------------------------------------------
    if lca_entropy >= entropy_floor:
        chosen_method, chosen_labels, chosen_k = "LCA", lca_series, lca_best_k
        reason = f"LCA entropy {lca_entropy:.3f} >= floor {entropy_floor} -> principled BIC selection used"
    else:
        chosen_method = method
        chosen_labels, chosen_k = trad_series, trad_k
        reason = f"LCA entropy {lca_entropy:.3f} < floor {entropy_floor} -> falling back to {method}"

    print(f"\nTraditional ({method}): elbow k = {trad_elbow_k}, fitted k = {trad_k}")
    print(f"LCA: BIC k = {lca_best_k}, entropy = {lca_entropy:.3f}")
    print(f"DECISION: {chosen_method} (k={chosen_k}) — {reason}")

    # -----------------------------------------------------------
    # E. Profile of the chosen clustering
    # -----------------------------------------------------------
    df_prof = df_proc.copy()
    df_prof["Cluster"] = chosen_labels
    print(f"\nCluster sizes:\n{df_prof['Cluster'].value_counts().sort_index()}")

    for col in raw_cols:
        if col in df_prof.columns:
            if pd.api.types.is_numeric_dtype(df_prof[col]) and df_prof[col].nunique() > 8:
                print(f"\n{col} mean per cluster:")
                print(df_prof.groupby("Cluster")[col].mean().round(2))
            else:
                print(f"\n{col} distribution per cluster:")
                print(pd.crosstab(df_prof["Cluster"], df_prof[col], normalize="index").round(2))

    return {
        "dimension": dimension_name,
        "chosen_method": chosen_method,
        "chosen_k": chosen_k,
        "labels": chosen_labels,
        "comparison_df": comparison_df,
        "trad_costs": trad_costs,
        "lca_scan_df": lca_scan_df,
    }


In [ ]:
# =====================================================
# REPORT GENERATOR — builds a readable summary from
# run_dimension()'s output. Call after run_dimension().
# =====================================================

import pandas as pd


def build_dimension_report(result, df_proc):
    """
    result: the dict returned by run_dimension()
    df_proc: your main processed dataframe
    Returns: a formatted string report, and a small summary DataFrame
    of cluster sizes/percentages for reuse elsewhere (e.g. Excel export).
    """
    dim = result["dimension"]
    method = result["chosen_method"]
    k = result["chosen_k"]
    labels = result["labels"]
    comparison_df = result["comparison_df"]

    raw_cols = cluster_models[dim]

    sizes = labels.value_counts().sort_index()
    total = len(labels)

    lines = []
    lines.append("=" * 78)
    lines.append(f"DIMENSION REPORT: {dim}")
    lines.append("=" * 78)

    lines.append(f"\nVariables used: {', '.join(raw_cols)}")
    lines.append(f"Method chosen: {method}  |  k = {k}")

    lc = comparison_df[comparison_df["Method"] == "LCA"].iloc[0]
    tc = comparison_df[comparison_df["Method"] != "LCA"].iloc[0]
    lines.append(f"LCA:         k={lc['k']}, {lc['Native_fit_metric']}")
    lines.append(f"Traditional: k={tc['k']}, {tc['Native_fit_metric']}")

    lines.append("\n--- Cluster sizes ---")
    size_rows = []
    for cid, n in sizes.items():
        pct = round(n / total * 100, 1)
        lines.append(f"  Cluster {cid}: n={n} ({pct}%)")
        size_rows.append({"Cluster": cid, "n": n, "Percent": pct})
    size_summary_df = pd.DataFrame(size_rows)

    lines.append("\n--- Variable profile per cluster ---")
    df_prof = df_proc.copy()
    df_prof["Cluster"] = labels

    for col in raw_cols:
        if col not in df_prof.columns:
            continue
        lines.append(f"\n{col}:")
        if pd.api.types.is_numeric_dtype(df_prof[col]) and df_prof[col].nunique() > 8:
            means = df_prof.groupby("Cluster")[col].mean().round(2)
            for cid, val in means.items():
                lines.append(f"  Cluster {cid}: mean = {val}")
        else:
            xtab = pd.crosstab(df_prof["Cluster"], df_prof[col], normalize="index").round(2)
            for cid in xtab.index:
                top = xtab.loc[cid].sort_values(ascending=False)
                top_str = ", ".join(f"{k}={v}" for k, v in top.head(3).items())
                lines.append(f"  Cluster {cid}: {top_str}")

    lines.append("\n--- Method comparison table ---")
    lines.append(comparison_df.to_string(index=False))

    report_text = "\n".join(lines)
    print(report_text)

    return report_text, size_summary_df


In [ ]:
# =====================================================
# RUN ALL DIMENSIONS
# =====================================================

all_results = {}
all_reports = {}
all_summaries = {}

for dim in cluster_models.keys():
    result = run_dimension(dim)
    report_text, size_summary_df = build_dimension_report(result, df_proc)

    all_results[dim] = result
    all_reports[dim] = report_text
    all_summaries[dim] = size_summary_df

    # Store the winning labels on df_proc immediately, generic column name for now —
    # you'll overwrite with a proper named mapping once you've reviewed each profile
    col_name = dim.replace(" & ", "_").replace(" ", "_") + "_Cluster_Auto"
    df_proc[col_name] = result["labels"]

    print("\n\n")

# =====================================================
# OVERVIEW — which method won for each dimension, at a glance
# =====================================================

overview_rows = []
for dim, result in all_results.items():
    overview_rows.append({
        "Dimension": dim,
        "Chosen Method": result["chosen_method"],
        "k": result["chosen_k"],
        "n Clusters (actual)": result["labels"].nunique(),
    })

overview_df = pd.DataFrame(overview_rows)
print("=" * 78)
print("OVERVIEW — method chosen per dimension")
print("=" * 78)
print(overview_df.to_string(index=False))

In [ ]:
def normalize_qid(x):
    s = str(x)
    return str(int(float(s))) if s.endswith('.0') else s

question_map = {
    normalize_qid(qid): text
    for qid, text in zip(df_pitanja["QuestionID"], df_pitanja["Text"])
}

option_map = {}
for qid, options in zip(df_pitanja["QuestionID"], df_pitanja["Options"]):
    qid = normalize_qid(qid)
    if pd.notnull(options):
        opts = {}
        for pair in str(options).split(';'):
            pair = pair.strip()
            if '=' in pair:
                code, label = pair.split('=', 1)
                opts[code.strip()] = label.strip()
        option_map[qid] = opts

def rename_column(col):
    """'2563_1' -> 'Ich fände es interessant... — Tipps, wie ich...'"""
    if '_' in col:
        prefix, code = col.split('_', 1)
    else:
        prefix, code = col, None
    qtext = question_map.get(prefix, prefix)
    if code is None:
        return qtext
    if code == "no_selection":
        return f"{qtext} — [no selection]"
    code_str = str(code)
    if code_str.endswith(".0"):
        code_str = code_str[:-2]
    decoded = option_map.get(prefix, {}).get(code_str)
    return f"{qtext} — {decoded if decoded else code}"

In [ ]:
# =====================================================
# COMBINED NAMING PROMPT — ALL DIMENSIONS AT ONCE
#
# Requires rename_column(), question_map, option_map to already
# exist (from your cell that builds them from df_pitanja) BEFORE
# this cell runs.
# =====================================================

def _option_label(col, code, option_map):
    """Decodes a crosstab CELL VALUE (not the column name) — used for
    select-one/ordinal variables where the value itself is a raw code
    (e.g. '341.2' = 3.0 -> 'Ich finde es gut...'). For multi-select
    dummy columns the cell value is just 0/1 and won't match anything
    in option_map, so it's returned as-is, which is correct."""
    if option_map is None:
        return code
    if "_" in col:
        base, opt_code = col.split("_", 1)
    else:
        base, opt_code = col, code
    opt_code_str = str(opt_code)
    if opt_code_str.endswith(".0"):
        opt_code_str = opt_code_str[:-2]
    decoded = option_map.get(base, {}).get(opt_code_str)
    return f"{code} ({decoded})" if decoded else code


def build_combined_naming_prompt(all_results, df_proc, question_map=None,
                                  option_map=None, top_n=3):
    """
    Builds ONE prompt covering every dimension in all_results, asking the
    AI to return a label dict + mapping line for each dimension in a single
    response. Prints only the prompt (no verbose per-variable console dump)
    to keep things short enough to paste in one go.

    Uses the shared rename_column() function (defined earlier in your
    notebook, from question_map/option_map) to decode column names —
    do not duplicate that logic here.
    """
    sections = []
    code_format_blocks = []

    for dim, result in all_results.items():
        raw_cols = cluster_models[dim]
        labels = result["labels"]
        df_prof = df_proc.copy()
        df_prof["Cluster"] = labels
        sizes = labels.value_counts().sort_index()
        total = len(labels)

        dim_key = dim.replace(" & ", "_").replace(" ", "_")
        dict_var = f"{dim_key}_labels"
        label_col = f"{dim_key}_Label"
        cluster_col = f"{dim_key}_Cluster_Auto"

        lines = [f"### DIMENSION: {dim}  (k={result['chosen_k']}, method={result['chosen_method']})"]
        lines.append("Cluster sizes: " + ", ".join(
            f"{cid}={n} ({round(n/total*100,1)}%)" for cid, n in sizes.items()
        ))

        for col in raw_cols:
            matched = [c for c in df_prof.columns if c == col or c.startswith(col + "_")]
            for c in matched:
                if c not in df_prof.columns:
                    continue
                qtext = rename_column(c)
                if pd.api.types.is_numeric_dtype(df_prof[c]) and df_prof[c].nunique() > 8:
                    means = df_prof.groupby("Cluster")[c].mean().round(2)
                    lines.append(f"{qtext} (mean): " + ", ".join(
                        f"C{cid}={val}" for cid, val in means.items()
                    ))
                else:
                    xtab = pd.crosstab(df_prof["Cluster"], df_prof[c], normalize="index").round(2)
                    row_strs = []
                    for cid in xtab.index:
                        top = xtab.loc[cid].sort_values(ascending=False).head(top_n)
                        top_str = "/".join(f"{_option_label(c, k, option_map)}={v}" for k, v in top.items())
                        row_strs.append(f"C{cid}:{top_str}")
                    lines.append(f"{qtext}: " + "  ".join(row_strs))

        sections.append("\n".join(lines))

        desc_var = f"{dim_key}_descriptions"
        code_format_blocks.append(
            f"{dict_var} = {{\n"
            + "".join(f'    {i}: "...",  # short 2-4 word name\n' for i in range(result["chosen_k"]))
            + "}\n"
            f"{desc_var} = {{\n"
            + "".join(f'    {i}: "...",  # one-sentence description\n' for i in range(result["chosen_k"]))
            + "}\n"
            f'df_proc["{label_col}"] = df_proc["{cluster_col}"].map({dict_var})'
        )

    prompt = (
        "I ran cluster analyses across several dimensions of a customer survey. "
        "For EACH dimension below, suggest a short (2-4 word) descriptive name "
        "and a one-sentence description for each cluster, capturing what makes "
        "it distinct from the others in that same dimension.\n\n"
        "Give me ALL results as ready-to-paste Python code ONLY — put the short "
        "name directly in the *_labels dict and the one-sentence description "
        "directly in the *_descriptions dict for each dimension below (do NOT "
        "also repeat the descriptions as separate prose paragraphs). Match "
        "these exact formats:\n\n"
        + "\n\n".join(code_format_blocks)
        + "\n\n"
        + "=" * 70 + "\nDATA\n" + "=" * 70 + "\n\n"
        + "\n\n".join(sections)
    )

    print(prompt)
    return prompt


# =====================================================
# USAGE
# =====================================================

# combined_prompt = build_combined_naming_prompt(all_results, df_proc,
#                                                  question_map=question_map,
#                                                  option_map=option_map)

In [ ]:
combined_prompt = build_combined_naming_prompt(all_results, df_proc,
                                                 question_map=question_map,
                                                 option_map=option_map)

In [ ]:
Advice_Style_labels = {
    0: "Self-Reliant",
    1: "Relationship-Seeking",
    2: "Passive but Reassured",
}
Advice_Style_descriptions = {
    0: "Rarely or never contacts an advisor and largely feels self-sufficient or unbothered by the lack of contact.",
    1: "Meets with an advisor more often than any other group and still wants more — values an ongoing personal relationship.",
    2: "Rarely engages with an advisor and doesn't want more contact, but values knowing a named contact exists if needed.",
}
df_proc["Advice_Style_Label"] = df_proc["Advice_Style_Cluster_Auto"].map(Advice_Style_labels)

Advice_Topics_of_Interest_labels = {
    0: "Retirement-Focused Savers",
    1: "Broadly Engaged Planners",
    2: "Digital-Curious",
    3: "No Topic Interest",
    4: "Cost & Digital Optimizers",
    5: "Spending-Focused",
    6: "Fee-Focused",
}
Advice_Topics_of_Interest_descriptions = {
    0: "Interested almost exclusively in saving for retirement, with little interest in other topics.",
    1: "Interested across nearly every topic offered — fees, retirement, digital tools, and budgeting all show above-average interest.",
    2: "Interested exclusively in making better use of the bank's digital offering.",
    3: "Selected no topics of interest at all — entirely disengaged from advisory topic offers.",
    4: "Interested in a mix of fee savings, spending control, and digital tools rather than any single dominant topic.",
    5: "Interested almost exclusively in reducing day-to-day spending.",
    6: "Interested almost exclusively in saving on fees and transaction costs.",
}
df_proc["Advice_Topics_of_Interest_Label"] = df_proc["Advice_Topics_of_Interest_Cluster_Auto"].map(Advice_Topics_of_Interest_labels)

Channel_Behavior_Preference_labels = {
    0: "Full-Spectrum Engaged",
    1: "Mobile-First",
    2: "Passive / No Preference",
    3: "Branch Loyalists",
    4: "Web Banking Loyalists",
    5: "Digital-First, Phone-for-Advice",
}
Channel_Behavior_Preference_descriptions = {
    0: "Balanced, moderate use across nearly every channel with no single dominant preference.",
    1: "Heavy app use with almost no online/PC banking; strongly prefers the app for information above all other channels.",
    2: "Shows no clear preferred channel for information, purchases, or advice — mostly answers 'not applicable' across preference questions.",
    3: "Visited a branch most recently of any group and strongly prefers in-person branch service for purchases and advice.",
    4: "Heavy online/PC banking use with low app and branch use; prefers web banking for information.",
    5: "Heavily digital in day-to-day banking, but distinctly prefers phone contact specifically for advice and purchases.",
}
df_proc["Channel_Behavior_Preference_Label"] = df_proc["Channel_Behavior_Preference_Cluster_Auto"].map(Channel_Behavior_Preference_labels)

Digital_Fintech_Engagement_labels = {
    0: "Fintech Power Users",
    1: "Fintech-Unaware",
    2: "Broadly Unaware",
    3: "Mainstream Digital Payers",
}
Digital_Fintech_Engagement_descriptions = {
    0: "Broadly engaged across direct/online banks and brokers, with above-average use of independent financial advisors.",
    1: "Largely unfamiliar with neobanks and crypto platforms, though moderately aware of direct banks and brokers.",
    2: "Shows the highest 'don't know' rate across nearly every fintech and advisory question — the least product-aware group overall.",
    3: "Uses mainstream payment apps but little else; low awareness or use of niche fintech products — the majority default profile.",
}
df_proc["Digital_Fintech_Engagement_Label"] = df_proc["Digital_Fintech_Engagement_Cluster_Auto"].map(Digital_Fintech_Engagement_labels)

Information_Contact_Openness_labels = {
    0: "Advisor-Centered, In-Person",
    1: "Broad Multi-Channel Engaged",
    2: "Other Sources, Split on Contact",
    3: "Independent Researchers, Contact-Averse",
    4: "Relationship & In-Person Driven",
    5: "Passive / No Selection",
    6: "Digital & Media Self-Researchers",
}
Information_Contact_Openness_descriptions = {
    0: "Relies primarily on their bank advisor for information and strongly prefers in-person contact for follow-up.",
    1: "Draws on multiple information sources at once and is open to a wide range of contact methods, including email.",
    2: "Relies on other/less common information sources and is evenly split on whether in-person contact appeals to them.",
    3: "Researches independently through moderate use of multiple sources but wants zero follow-up contact of any kind.",
    4: "Relies on family/friends and independent advisors for information and wants in-person contact exclusively.",
    5: "Selected no information sources and no preferred contact methods — the most passive, disengaged group.",
    6: "Researches through the bank's own digital channels and general media rather than personal or independent sources.",
}
df_proc["Information_Contact_Openness_Label"] = df_proc["Information_Contact_Openness_Cluster_Auto"].map(Information_Contact_Openness_labels)

Membership_Engagement_labels = {
    0: "Non-Members",
    1: "Membership Not Applicable",
    2: "Members",
}
Membership_Engagement_descriptions = {
    0: "Explicitly not a cooperative member/shareholder of the bank.",
    1: "Membership status is not applicable to this respondent.",
    2: "A cooperative member/shareholder of the bank.",
}
df_proc["Membership_Engagement_Label"] = df_proc["Membership_Engagement_Cluster_Auto"].map(Membership_Engagement_labels)

In [ ]:
profile_cols = [
    "Advice_Style_Label",
    "Advice_Topics_of_Interest_Label",
    "Channel_Behavior_Preference_Label",
    "Digital_Fintech_Engagement_Label",
    "Information_Contact_Openness_Label",
    "Membership_Engagement_Label",
    "930",  # raw Values category
]

profile_df = df_proc[profile_cols].copy()
profile_df.columns = ["Advice_Style", "Advice_Topics", "Channel", "Digital", "Info_Contact", "Membership", "Values"]

print(profile_df.head(10))
print("\nShape:", profile_df.shape)

In [ ]:
from scipy.stats import chi2_contingency
import itertools

def cramers_v(x, y):
    contingency = pd.crosstab(x, y)
    chi2 = chi2_contingency(contingency)[0]
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    return (chi2 / (n * min_dim)) ** 0.5

cols = profile_df.columns.tolist()
results = []

for col1, col2 in itertools.combinations(cols, 2):
    v = cramers_v(profile_df[col1], profile_df[col2])
    results.append({"Dimension 1": col1, "Dimension 2": col2, "Cramér's V": round(v, 3)})

assoc_df = pd.DataFrame(results).sort_values("Cramér's V", ascending=False)
print(assoc_df.to_string(index=False))

In [ ]:
import prince
import matplotlib.pyplot as plt

profile_mca_input = profile_df.astype(str)

mca = prince.MCA(n_components=2, random_state=42)
mca = mca.fit(profile_mca_input)

# Respondent map
coords = mca.transform(profile_mca_input)

plt.figure(figsize=(9, 7))
plt.scatter(coords.iloc[:, 0], coords.iloc[:, 1], alpha=0.15, s=10)
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.title("MCA map of full customer profiles")
plt.show()

# Category map
category_coords = mca.column_coordinates(profile_mca_input)
print(category_coords.sort_values(0))

# Separately, check how much variance the 2 components actually explain
print("\nVariance explained:")
print(mca.eigenvalues_summary)

In [ ]:
from stepmix import StepMix
from sklearn.preprocessing import LabelEncoder
import pandas as pd

X_profile = profile_df.copy()
for c in X_profile.columns:
    X_profile[c] = LabelEncoder().fit_transform(X_profile[c].astype(str))
X_profile_encoded = X_profile.to_numpy()

results = []
for k in range(2, 11):
    model = StepMix(n_components=k, measurement="categorical", random_state=42, n_init=10)
    model.fit(X_profile_encoded)
    results.append({"k": k, "bic": model.bic(X_profile_encoded), "aic": model.aic(X_profile_encoded)})

lca_profile_results = pd.DataFrame(results)
print(lca_profile_results)

In [ ]:
best_k_final = int(lca_profile_results.loc[lca_profile_results["bic"].idxmin(), "k"])
print("Best k by BIC:", best_k_final)

final_lca = StepMix(n_components=best_k_final, measurement="categorical", random_state=42, n_init=25)
final_lca.fit(X_profile_encoded)
print("Relative entropy:", round(final_lca.relative_entropy(X_profile_encoded), 3))

type_labels = final_lca.predict(X_profile_encoded)
profile_df["Final_Type"] = type_labels
df_proc["Final_Type"] = profile_df["Final_Type"]

print(profile_df["Final_Type"].value_counts().sort_index())

In [ ]:
# =====================================================
# FINAL TYPOLOGY NAMING PROMPT
# Same pattern as build_combined_naming_prompt, but one level up:
# uses profile_df's dimension-labels as the "variables" instead of
# raw survey columns.
# =====================================================

def build_final_type_naming_prompt(df_proc, profile_df, type_col="Final_Type",
                                    profile_cols=None, top_n=3):
    if profile_cols is None:
        profile_cols = ["Advice_Style", "Advice_Topics", "Channel",
                         "Digital", "Info_Contact", "Membership", "Values"]

    types = df_proc[type_col]
    sizes = types.value_counts().sort_index()
    total = len(types)
    k = types.nunique()

    lines = [f"### FINAL TYPOLOGY  (k={k})"]
    lines.append("Type sizes: " + ", ".join(
        f"{tid}={n} ({round(n/total*100,1)}%)" for tid, n in sizes.items()
    ))

    for col in profile_cols:
        if col not in profile_df.columns:
            continue
        xtab = pd.crosstab(df_proc[type_col], profile_df[col], normalize="index").round(2)
        row_strs = []
        for tid in xtab.index:
            top = xtab.loc[tid].sort_values(ascending=False).head(top_n)
            top_str = "/".join(f"{cat}={v}" for cat, v in top.items())
            row_strs.append(f"T{tid}:{top_str}")
        lines.append(f"{col}: " + "  ".join(row_strs))

    data_section = "\n".join(lines)

    code_format = (
        "final_type_names = {\n"
        + "".join(f'    {i}: "...",  # short 2-4 word name\n' for i in range(k))
        + "}\n"
        "final_type_descriptions = {\n"
        + "".join(f'    {i}: "...",  # one-sentence description\n' for i in range(k))
        + "}\n"
        f'df_proc["Final_Type_Label"] = df_proc["{type_col}"].map(final_type_names)'
    )

    prompt = (
        "I ran a final-stage cluster analysis combining several already-named "
        "customer dimensions (Advice_Style, Advice_Topics, Channel, Digital, "
        "Info_Contact, Membership, Values) into an overall customer typology "
        f"with {k} final types. For EACH type below, suggest a short (2-5 word) "
        "descriptive name and a one-sentence description that captures what "
        "combination of dimension-level traits makes it distinct from the "
        "other final types.\n\n"
        "Give me the result as ready-to-paste Python code ONLY, in exactly "
        "this format:\n\n"
        + code_format
        + "\n\n"
        + "=" * 70 + "\nDATA\n" + "=" * 70 + "\n\n"
        + data_section
    )

    print(prompt)
    return prompt


# =====================================================
# USAGE
# =====================================================

# final_type_prompt = build_final_type_naming_prompt(df_proc, profile_df)

In [ ]:
final_type_prompt = build_final_type_naming_prompt(df_proc, profile_df)

In [ ]:
final_type_names = {
    0: "Engaged Members, Broad Contact",
    1: "Passive Members, Low Interest",
    2: "Contact-Averse Non-Members",
    3: "Digital Self-Researchers, Mixed Membership",
}
final_type_descriptions = {
    0: "The largest group and mostly bank members; combines relationship-seeking advice preferences, retirement-focused interests, and the broadest use of both branch and multi-channel contact of any type — the most engaged, in-person-friendly segment.",
    1: "Overwhelmingly bank members but the most passive type overall — the highest share with no topic interest at all, no clear channel preference, and low-engagement information habits despite the strong membership relationship.",
    2: "Defined almost entirely by wanting zero follow-up contact (83% Independent Researchers, Contact-Averse) and by membership status mostly being not applicable — the least internally consistent type, and the one most dependent on membership status for its distinctiveness rather than a robust standalone behavioral pattern.",
    3: "Evenly split between members and non-members, with the heaviest digital self-research habits, the strongest phone-for-advice and mobile-first channel behavior, and the highest share of fintech power users of any type.",
}
df_proc["Final_Type_Label"] = df_proc["Final_Type"].map(final_type_names)

In [ ]:
def full_dimension_report(df, cluster_col, cluster_prefixes, labels, descriptions, question_map, option_map):
    print("=" * 100)
    print(f"DIMENSION: {cluster_col}")
    print("=" * 100)

    print("\nQuestions used to build this dimension:")
    for prefix in cluster_prefixes:
        qtext = question_map.get(normalize_qid(prefix), prefix)
        print(f"  {prefix}: {qtext}")

    print("\nCluster sizes:")
    sizes = df[cluster_col].value_counts().sort_index()
    for cid, n in sizes.items():
        print(f"  {cid} — {labels[cid]}: n={n} ({n/len(df)*100:.1f}%)")

    print("\nCluster descriptions:")
    for cid in sorted(labels.keys()):
        print(f"  [{labels[cid]}] {descriptions[cid]}")

    matched_cols = []
    for prefix in cluster_prefixes:
        matched_cols += [c for c in df.columns if c == prefix or c.startswith(prefix + "_")]

    numeric_cols = [c for c in matched_cols if pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() > 8]
    categorical_cols_here = [c for c in matched_cols if c not in numeric_cols]

    numeric_profile = None
    if numeric_cols:
        numeric_profile = df.groupby(cluster_col)[numeric_cols].mean().round(2)
        numeric_profile.columns = [rename_column(c) for c in numeric_profile.columns]
        numeric_profile.index = [f"{i} — {labels[i]}" for i in numeric_profile.index]
        print(f"\nNumeric profile table for {cluster_col}:")
        display(numeric_profile)

    categorical_profiles = {}
    for c in categorical_cols_here:
        xtab = pd.crosstab(df[cluster_col], df[c], normalize="index").round(2)
        xtab.index = [f"{i} — {labels[i]}" for i in xtab.index]
        categorical_profiles[rename_column(c)] = xtab
        print(f"\n{rename_column(c)} — distribution per cluster:")
        display(xtab)

    return {"numeric": numeric_profile, "categorical": categorical_profiles}

In [ ]:
adv_profile = full_dimension_report(df_proc, "Advice_Style_Cluster_Auto", cluster_models["Advice Style"],
                                     Advice_Style_labels, Advice_Style_descriptions, question_map, option_map)

topics_profile = full_dimension_report(df_proc, "Advice_Topics_of_Interest_Cluster_Auto", cluster_models["Advice Topics of Interest"],
                                        Advice_Topics_of_Interest_labels, Advice_Topics_of_Interest_descriptions, question_map, option_map)

channel_profile = full_dimension_report(df_proc, "Channel_Behavior_Preference_Cluster_Auto", cluster_models["Channel Behavior & Preference"],
                                         Channel_Behavior_Preference_labels, Channel_Behavior_Preference_descriptions, question_map, option_map)

digital_profile = full_dimension_report(df_proc, "Digital_Fintech_Engagement_Cluster_Auto", cluster_models["Digital & Fintech Engagement"],
                                         Digital_Fintech_Engagement_labels, Digital_Fintech_Engagement_descriptions, question_map, option_map)

info_profile = full_dimension_report(df_proc, "Information_Contact_Openness_Cluster_Auto", cluster_models["Information & Contact Openness"],
                                      Information_Contact_Openness_labels, Information_Contact_Openness_descriptions, question_map, option_map)

membership_profile = full_dimension_report(df_proc, "Membership_Engagement_Cluster_Auto", cluster_models["Membership Engagement"],
                                            Membership_Engagement_labels, Membership_Engagement_descriptions, question_map, option_map)

In [ ]:
print("=" * 100)
print("FINAL TYPOLOGY")
print("=" * 100)

sizes = df_proc["Final_Type"].value_counts().sort_index()
for cid, n in sizes.items():
    print(f"  {cid} — {final_type_names[cid]}: n={n} ({n/len(df_proc)*100:.1f}%)")

print("\nFinal type descriptions:")
for cid in sorted(final_type_names.keys()):
    print(f"  [{final_type_names[cid]}] {final_type_descriptions[cid]}")

print("\nHow each final type performs on the 7-dimension profile vector:")
for col in ["Advice_Style", "Advice_Topics", "Channel", "Digital", "Info_Contact", "Membership", "Values"]:
    print(f"\n--- {col} ---")
    display((pd.crosstab(df_proc["Final_Type"], profile_df[col], normalize="index") * 100).round(1))

In [ ]:
import pandas as pd

output_path = "cluster_typology_report.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    dimension_data = {
        "Advice Style": (Advice_Style_labels, Advice_Style_descriptions, "Advice_Style_Cluster_Auto", adv_profile),
        "Advice Topics": (Advice_Topics_of_Interest_labels, Advice_Topics_of_Interest_descriptions, "Advice_Topics_of_Interest_Cluster_Auto", topics_profile),
        "Channel": (Channel_Behavior_Preference_labels, Channel_Behavior_Preference_descriptions, "Channel_Behavior_Preference_Cluster_Auto", channel_profile),
        "Digital Fintech": (Digital_Fintech_Engagement_labels, Digital_Fintech_Engagement_descriptions, "Digital_Fintech_Engagement_Cluster_Auto", digital_profile),
        "Info Contact": (Information_Contact_Openness_labels, Information_Contact_Openness_descriptions, "Information_Contact_Openness_Cluster_Auto", info_profile),
        "Membership": (Membership_Engagement_labels, Membership_Engagement_descriptions, "Membership_Engagement_Cluster_Auto", membership_profile),
    }

    for sheet_name, (labels, descriptions, cluster_col, profile) in dimension_data.items():
        sizes = df_proc[cluster_col].value_counts().sort_index()
        summary_rows = []
        for cid in sorted(labels.keys()):
            summary_rows.append({
                "Cluster": cid,
                "Name": labels[cid],
                "n": sizes.get(cid, 0),
                "Percent": round(sizes.get(cid, 0) / len(df_proc) * 100, 1),
                "Description": descriptions[cid],
            })
        summary_df = pd.DataFrame(summary_rows)
        summary_df.to_excel(writer, sheet_name=sheet_name[:31], index=False, startrow=0)
        current_row = len(summary_df) + 3

        if profile["numeric"] is not None:
            profile["numeric"].to_excel(writer, sheet_name=sheet_name[:31], startrow=current_row)
            current_row += len(profile["numeric"]) + 3

        for var_name, xtab in profile["categorical"].items():
            label_df = pd.DataFrame({f"--- {var_name} ---": []})
            label_df.to_excel(writer, sheet_name=sheet_name[:31], startrow=current_row, index=False)
            current_row += 2
            xtab.to_excel(writer, sheet_name=sheet_name[:31], startrow=current_row)
            current_row += len(xtab) + 3

    final_sizes = df_proc["Final_Type"].value_counts().sort_index()
    final_summary_rows = []
    for cid in sorted(final_type_names.keys()):
        final_summary_rows.append({
            "Type": cid,
            "Name": final_type_names[cid],
            "n": final_sizes.get(cid, 0),
            "Percent": round(final_sizes.get(cid, 0) / len(df_proc) * 100, 1),
            "Description": final_type_descriptions[cid],
        })
    final_summary_df = pd.DataFrame(final_summary_rows)
    final_summary_df.to_excel(writer, sheet_name="Final Typology", index=False, startrow=0)

    current_row = len(final_summary_df) + 3
    for col in ["Advice_Style", "Advice_Topics", "Channel", "Digital", "Info_Contact", "Membership", "Values"]:
        crosstab = (pd.crosstab(df_proc["Final_Type"], profile_df[col], normalize="index") * 100).round(1)
        label_df = pd.DataFrame({f"--- {col} ---": []})
        label_df.to_excel(writer, sheet_name="Final Typology", startrow=current_row, index=False)
        current_row += 2
        crosstab.to_excel(writer, sheet_name="Final Typology", startrow=current_row)
        current_row += len(crosstab) + 3

    df_proc.to_excel(writer, sheet_name="Full Dataset", index=False)

print(f"Saved: {output_path}")

In [ ]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>